In [1]:
!pip install -q \
langchain \
langchain-community \
sentence-transformers \
faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
import langchain
import faiss
import sentence_transformers

print("LangChain:", langchain.__version__)
print("FAISS Loaded")
print("Sentence Transformers Loaded")

LangChain: 1.3.6
FAISS Loaded
Sentence Transformers Loaded


In [3]:
text = """
LangChain is an open-source framework for building applications powered by large language models.

RAG stands for Retrieval Augmented Generation.

Embeddings convert text into numerical vectors.

FAISS is a vector database used for similarity search.
"""

print(text)


LangChain is an open-source framework for building applications powered by large language models.

RAG stands for Retrieval Augmented Generation.

Embeddings convert text into numerical vectors.

FAISS is a vector database used for similarity search.



In [4]:
from langchain_core.documents import Document

doc = Document(page_content=text)

print(doc)

page_content='
LangChain is an open-source framework for building applications powered by large language models.

RAG stands for Retrieval Augmented Generation.

Embeddings convert text into numerical vectors.

FAISS is a vector database used for similarity search.
'


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20
)

chunks = splitter.split_documents([doc])

print("Number of chunks:", len(chunks))

Number of chunks: 3


In [6]:
for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1}")
    print(chunk.page_content)


Chunk 1
LangChain is an open-source framework for building applications powered by large language models.

Chunk 2
RAG stands for Retrieval Augmented Generation.

Embeddings convert text into numerical vectors.

Chunk 3
FAISS is a vector database used for similarity search.


In [7]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")

/tmp/ipykernel_799/173910916.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings
/tmp/ipykernel_799/173910916.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your setting

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded


In [8]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    chunks,
    embedding_model
)

print("Vector Store Created")

Vector Store Created


In [9]:
retriever = vectorstore.as_retriever()

print("Retriever Ready")

Retriever Ready


In [10]:
query = "What is RAG?"

results = retriever.invoke(query)

results

[Document(id='e558188a-9e11-4b91-ae06-ceb94b6f2d6d', metadata={}, page_content='RAG stands for Retrieval Augmented Generation.\n\nEmbeddings convert text into numerical vectors.'),
 Document(id='b31e9b04-6903-4d62-83b7-765a5f9b2dc8', metadata={}, page_content='LangChain is an open-source framework for building applications powered by large language models.'),
 Document(id='9485ac36-b1fe-4da0-8e8e-d4b30d31cbc5', metadata={}, page_content='FAISS is a vector database used for similarity search.')]

In [11]:
for doc in results:
    print(doc.page_content)

RAG stands for Retrieval Augmented Generation.

Embeddings convert text into numerical vectors.
LangChain is an open-source framework for building applications powered by large language models.
FAISS is a vector database used for similarity search.


In [12]:
context = results[0].page_content

question = "What is RAG?"

prompt = f"""
Answer the question using the context below.

Context:
{context}

Question:
{question}
"""

print(prompt)


Answer the question using the context below.

Context:
RAG stands for Retrieval Augmented Generation.

Embeddings convert text into numerical vectors.

Question:
What is RAG?

